# Fig. 1E, fig. S1G: exemplar genes (Hnrnpa2b1 + Qk)

Full-width panel showing the #1 (shortening) and #2 (lengthening) composite-ranked
genes. For each gene: stacked-area of 3'UTR-site proportions along pseudotime (with
a cell-type annotation bar coloured by `paper_palette_celltypes`) and the per-cell
UTR-length index W(t) trajectory with SEM ribbon.

- Kernel: `apa-python`.
- Inputs: `UTR_count_table_genelevel_harmony_proportions_lineage_smoothed.h5ad`
  - `DirMult_optimized/composite_ranking_optimized2_df5.csv`.
- Output: `$M26_RESULTS_ROOT/Fig1/fig1e_exemplars/`.


## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")

In [ ]:
import sys, warnings

warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import rankdata

STYLE_DIR = f"{REPO_ROOT}/figures/style"
if STYLE_DIR not in sys.path:
    sys.path.insert(0, STYLE_DIR)
import paper_fig_style as S

S.apply_rcparams()

PANEL = "fig1e_exemplars"
PANEL_W_PX, PANEL_H_PX = S.FULL_W_PX, S.mm_to_px(50)  # 174 x 50 mm (full-grid)
FIGSIZE = (PANEL_W_PX / 300, PANEL_H_PX / 300)

BASE = Path(f"{DATA_ROOT}/3utr/wildtypes_analysis/2026_analyses/utrcalling")
UTR_SMOOTHED_PATH = BASE / "UTR_count_table_genelevel_harmony_proportions_lineage_smoothed.h5ad"
RANKING_PATH = BASE / "DirMult_optimized" / "composite_ranking_optimized2_df5.csv"

In [ ]:
def _get_gene_data(adata, gene_name):
    """Extract smoothed proportions for a gene by gene_name."""
    gene_mask = adata.var["gene_name"] == gene_name
    if gene_mask.sum() == 0:
        return None, None, None, None
    sub = adata[:, gene_mask]
    props = sub.layers["smoothed_proportions"]
    if hasattr(props, "toarray"):
        props = props.toarray()
    props = np.asarray(props, dtype=np.float64)
    utr_sizes = sub.var["utr_size"].values.astype(float)
    site_names = sub.var["UTR_site"].values.tolist()
    pt = adata.obs["dpt_pseudotime"].values.copy()
    return props, utr_sizes, site_names, pt


def _bin_pseudotime(pt, props, n_bins=40):
    """Bin cells by pseudotime; return bin_centres, mean_props, sem_props, counts."""
    valid = ~np.isnan(pt)
    pt_v, props_v = pt[valid], props[valid]
    bins = pd.cut(pt_v, bins=n_bins, labels=False)
    bin_centres = np.array([pt_v[bins == b].mean() for b in range(n_bins)])
    mean_props = np.zeros((n_bins, props_v.shape[1]))
    sem_props = np.zeros_like(mean_props)
    counts = np.zeros(n_bins)
    for b in range(n_bins):
        mask = bins == b
        n = mask.sum()
        counts[b] = n
        if n > 0:
            mean_props[b] = props_v[mask].mean(axis=0)
            if n > 1:
                sem_props[b] = props_v[mask].std(axis=0, ddof=1) / np.sqrt(n)
    row_sums = mean_props.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1
    mean_props = mean_props / row_sums
    keep = counts > 0
    return bin_centres[keep], mean_props[keep], sem_props[keep], counts[keep]


def _utr_colormap(k):
    """Colormap for k UTR sites (proximal -> warm, distal -> cool)."""
    if k <= 2:
        return ["#E24A33", "#4878CF"][:k]
    return list(sns.color_palette("RdYlBu_r", k))

In [ ]:
df = pd.read_csv(RANKING_PATH)
print("loading smoothed proportions AnnData...")
adata_smooth = sc.read_h5ad(UTR_SMOOTHED_PATH)
print(f"smoothed AnnData: {adata_smooth.shape} | layers: {list(adata_smooth.layers.keys())}")

EXEMPLAR_GENES = {
    "Hnrnpa2b1": {
        "delta_psi_str": "−0.38",
        "padj_str": "≈ 0",
        "direction": "Shortening",
        "rank": "#1",
        "note": "hnRNP RNA-binding protein",
    },
    "Qk": {
        "delta_psi_str": "+0.31",
        "padj_str": "2.7 × 10⁻²⁵³",
        "direction": "Lengthening",
        "rank": "#2",
        "note": "ASD / myelination",
    },
}

for gn, info in EXEMPLAR_GENES.items():
    row = df[df["gene_name"] == gn].iloc[0]
    print(
        f"{gn} (rank {info['rank']}): padj={row['padj']:.2e}, "
        f"d_psi={row['delta_psi']:+.4f}, K={row['n_sites']:.0f}, direction={row['direction']}"
    )

In [ ]:
fig, axes = plt.subplots(
    1,
    4,
    figsize=FIGSIZE,
    gridspec_kw={"width_ratios": [1.2, 1, 1.2, 1], "wspace": 0.35},
)

CELLTYPE_ORDER = S.MOUSE_LINEAGE_ORDER
df_obs_smooth = adata_smooth.obs[["dpt_pseudotime", "celltype"]].dropna(subset=["dpt_pseudotime"])

for gene_idx, (gene_name, info) in enumerate(EXEMPLAR_GENES.items()):
    ax_stack = axes[gene_idx * 2]
    ax_wt = axes[gene_idx * 2 + 1]

    props, utr_sizes, site_names, pt = _get_gene_data(adata_smooth, gene_name)
    assert props is not None, f"{gene_name} not found in smoothed AnnData"

    order = np.argsort(utr_sizes)
    utr_sizes = utr_sizes[order]
    site_names = [site_names[i] for i in order]
    props = props[:, order]
    K = props.shape[1]
    colors = _utr_colormap(K)

    bin_centres, mean_props, _, bin_counts = _bin_pseudotime(pt, props, n_bins=40)

    # stacked area of UTR-site proportions
    ax_stack.stackplot(bin_centres, mean_props.T, labels=site_names, colors=colors, alpha=0.85)
    ax_stack.set_ylim(-0.14, 1.0)
    ax_stack.set_xlim(bin_centres.min(), bin_centres.max())
    ax_stack.set_ylabel("Proportion", fontsize=S.FONT_LABEL)

    dir_col = S.COL_SHORT if info["direction"] == "Shortening" else S.COL_LONG
    ax_stack.set_title(
        f"$\\it{{{gene_name}}}$  ({info['direction']})",
        fontsize=S.FONT_LABEL,
        pad=4,
        color=dir_col,
        fontweight="bold",
    )
    ax_stack.legend(
        loc="upper left",
        fontsize=S.FONT_LEGEND - 0.5,
        framealpha=0.7,
        title=f"K = {K}",
        title_fontsize=S.FONT_LEGEND_TITLE - 0.5,
        handlelength=1,
        handleheight=0.7,
    )

    # cell-type annotation bar below the stacked area
    bar_y0, bar_y1 = -0.13, -0.02
    for ct in CELLTYPE_ORDER:
        if ct not in df_obs_smooth["celltype"].values:
            continue
        ct_pt = df_obs_smooth.loc[df_obs_smooth["celltype"] == ct, "dpt_pseudotime"]
        x0, x1 = float(ct_pt.min()), float(ct_pt.max())
        ax_stack.fill_between(
            [x0, x1],
            bar_y0,
            bar_y1,
            color=S.paper_palette_celltypes[ct],
            alpha=0.9,
            linewidth=0,
        )
    ax_stack.axhline(0, color="grey", linewidth=0.5, alpha=0.3)
    ax_stack.set_xticks([])
    ax_stack.tick_params(axis="x", bottom=False, labelbottom=False)
    ax_stack.spines["bottom"].set_visible(False)

    # cell-type legend below (only under the left gene)
    if gene_idx == 0:
        present = [ct for ct in CELLTYPE_ORDER if ct in df_obs_smooth["celltype"].values]
        ct_handles = [
            plt.Rectangle((0, 0), 1, 1, fc=S.paper_palette_celltypes[ct], alpha=0.9, linewidth=0)
            for ct in present
        ]
        fig.legend(
            ct_handles,
            present,
            loc="lower center",
            bbox_to_anchor=(0.5, 0.01),
            ncol=len(present),
            fontsize=S.FONT_LEGEND - 0.5,
            frameon=False,
            handlelength=1.0,
            handleheight=0.6,
            columnspacing=0.6,
        )
        ax_stack.legend(
            loc="upper left",
            fontsize=S.FONT_LEGEND - 0.5,
            framealpha=0.7,
            title=f"K = {K}",
            title_fontsize=S.FONT_LEGEND_TITLE - 0.5,
            handlelength=1,
            handleheight=0.7,
        )

    # W(t) trajectory
    ranks = (rankdata(utr_sizes, method="average") - 1) / (K - 1) if K > 1 else np.array([0.5])
    valid = ~np.isnan(pt)
    pt_v, props_v = pt[valid], props[valid]
    bins_idx = pd.cut(pt_v, bins=40, labels=False)
    W_bins, W_means, W_sems = [], [], []
    for b in range(40):
        mask = bins_idx == b
        n = mask.sum()
        if n > 0:
            W_cell = props_v[mask] @ ranks
            W_bins.append(pt_v[mask].mean())
            W_means.append(W_cell.mean())
            W_sems.append(W_cell.std(ddof=1) / np.sqrt(n) if n > 1 else 0)
    W_bins = np.array(W_bins)
    W_means = np.array(W_means)
    W_sems = np.array(W_sems)

    ax_wt.fill_between(W_bins, W_means - W_sems, W_means + W_sems, alpha=0.25, color=dir_col)
    ax_wt.plot(W_bins, W_means, "-o", color=dir_col, markersize=2, linewidth=1.2)
    ax_wt.annotate(
        "",
        xy=(W_bins[-1], W_means[-1]),
        xytext=(W_bins[0], W_means[0]),
        arrowprops=dict(arrowstyle="->", color=dir_col, lw=1, alpha=0.4),
    )
    ax_wt.text(
        0.97,
        0.95,
        f"Δψ = {info['delta_psi_str']}\npadj {info['padj_str']}\nRank {info['rank']}",
        transform=ax_wt.transAxes,
        ha="right",
        va="top",
        fontsize=S.FONT_ANNOT - 0.5,
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="0.7", boxstyle="round,pad=0.3"),
    )
    ax_wt.set_xlabel("Pseudotime", fontsize=S.FONT_LABEL)
    ax_wt.set_ylabel("W(t)" if gene_idx == 0 else "", fontsize=S.FONT_LABEL)
    ax_wt.tick_params(labelsize=S.FONT_TICK)

fig.subplots_adjust(left=0.04, right=0.98, top=0.88, bottom=0.16)
S.save_and_show(fig, PANEL)

### Alternative: equal-cell (quantile) pseudotime axis

The panel above bins pseudotime by equal width, which crushes the cell-dense qNSC/NB regions because the sparse aNSC clusters span a large pseudotime range. The version below bins by equal cell count (empirical-CDF transform of pseudotime), giving a uniform cell density on the x-axis while preserving the smoothed progression.


In [ ]:
# --- Alternative panel: equal-cell (quantile) pseudotime axis ---
# The default panel bins pseudotime in equal width, so the cell-sparse aNSC region (few
# cells spanning much of pseudotime) is stretched and the cell-dense qNSC/NB regions are
# crushed. Here cells are binned in equal count: pseudotime is transformed by its empirical
# CDF (rank/N) so cells are uniformly distributed on x, preserving resolution where cells
# are dense while keeping the smoothed-progression feel.
PANEL_U = "fig1e_exemplars__uniform"
N_BINS = 40

pt_all = adata_smooth.obs["dpt_pseudotime"].values.astype(float)
valid_all = ~np.isnan(pt_all)
u_all = np.full_like(pt_all, np.nan)
u_all[valid_all] = (rankdata(pt_all[valid_all], method="average") - 0.5) / valid_all.sum()
obs_u = adata_smooth.obs[["celltype"]].copy()
obs_u["u"] = u_all

fig, axes = plt.subplots(
    1,
    4,
    figsize=FIGSIZE,
    gridspec_kw={"width_ratios": [1.2, 1, 1.2, 1], "wspace": 0.35},
)
CELLTYPE_ORDER = S.MOUSE_LINEAGE_ORDER

for gene_idx, (gene_name, info) in enumerate(EXEMPLAR_GENES.items()):
    ax_stack = axes[gene_idx * 2]
    ax_wt = axes[gene_idx * 2 + 1]

    props, utr_sizes, site_names, _ = _get_gene_data(adata_smooth, gene_name)
    order = np.argsort(utr_sizes)
    utr_sizes = utr_sizes[order]
    site_names = [site_names[i] for i in order]
    props = props[:, order]
    K = props.shape[1]
    colors = _utr_colormap(K)

    bin_centres, mean_props, _, _ = _bin_pseudotime(u_all, props, n_bins=N_BINS)
    ax_stack.stackplot(bin_centres, mean_props.T, labels=site_names, colors=colors, alpha=0.85)
    ax_stack.set_ylim(-0.14, 1.0)
    ax_stack.set_xlim(0, 1)
    ax_stack.set_ylabel("Proportion", fontsize=S.FONT_LABEL)
    dir_col = S.COL_SHORT if info["direction"] == "Shortening" else S.COL_LONG
    ax_stack.set_title(
        f"$\\it{{{gene_name}}}$  ({info['direction']})",
        fontsize=S.FONT_LABEL,
        pad=4,
        color=dir_col,
        fontweight="bold",
    )
    ax_stack.legend(
        loc="upper left",
        fontsize=S.FONT_LEGEND - 0.5,
        framealpha=0.7,
        title=f"K = {K}",
        title_fontsize=S.FONT_LEGEND_TITLE - 0.5,
        handlelength=1,
        handleheight=0.7,
    )

    bar_y0, bar_y1 = -0.13, -0.02
    for ct in CELLTYPE_ORDER:
        cu = obs_u.loc[obs_u["celltype"] == ct, "u"].dropna()
        if len(cu) == 0:
            continue
        ax_stack.fill_between(
            [float(cu.min()), float(cu.max())],
            bar_y0,
            bar_y1,
            color=S.paper_palette_celltypes[ct],
            alpha=0.9,
            linewidth=0,
        )
    ax_stack.axhline(0, color="grey", linewidth=0.5, alpha=0.3)
    ax_stack.set_xticks([0, 0.5, 1.0])
    ax_stack.tick_params(axis="x", labelsize=S.FONT_TICK)
    ax_stack.set_xlabel("Pseudotime (equal-cell quantile)", fontsize=S.FONT_LABEL)

    if gene_idx == 0:
        present = [ct for ct in CELLTYPE_ORDER if (obs_u["celltype"] == ct).any()]
        ct_handles = [
            plt.Rectangle((0, 0), 1, 1, fc=S.paper_palette_celltypes[ct], alpha=0.9, linewidth=0)
            for ct in present
        ]
        fig.legend(
            ct_handles,
            present,
            loc="lower center",
            bbox_to_anchor=(0.5, 0.01),
            ncol=len(present),
            fontsize=S.FONT_LEGEND - 0.5,
            frameon=False,
            handlelength=1.0,
            handleheight=0.6,
            columnspacing=0.6,
        )

    ranks = (rankdata(utr_sizes, method="average") - 1) / (K - 1) if K > 1 else np.array([0.5])
    valid = ~np.isnan(u_all)
    u_v, props_v = u_all[valid], props[valid]
    bins_idx = pd.cut(u_v, bins=N_BINS, labels=False)
    Wb, Wm, Ws = [], [], []
    for b in range(N_BINS):
        mask = bins_idx == b
        n = mask.sum()
        if n > 0:
            Wc = props_v[mask] @ ranks
            Wb.append(u_v[mask].mean())
            Wm.append(Wc.mean())
            Ws.append(Wc.std(ddof=1) / np.sqrt(n) if n > 1 else 0)
    Wb, Wm, Ws = map(np.array, (Wb, Wm, Ws))
    ax_wt.fill_between(Wb, Wm - Ws, Wm + Ws, alpha=0.25, color=dir_col)
    ax_wt.plot(Wb, Wm, "-o", color=dir_col, markersize=2, linewidth=1.2)
    ax_wt.text(
        0.97,
        0.95,
        f"Δψ = {info['delta_psi_str']}\npadj {info['padj_str']}\nRank {info['rank']}",
        transform=ax_wt.transAxes,
        ha="right",
        va="top",
        fontsize=S.FONT_ANNOT - 0.5,
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="0.7", boxstyle="round,pad=0.3"),
    )
    ax_wt.set_xlabel("Pseudotime (equal-cell quantile)", fontsize=S.FONT_LABEL)
    ax_wt.set_ylabel("W(t)" if gene_idx == 0 else "", fontsize=S.FONT_LABEL)
    ax_wt.set_xlim(0, 1)
    ax_wt.tick_params(labelsize=S.FONT_TICK)

fig.subplots_adjust(left=0.04, right=0.98, top=0.88, bottom=0.16)
S.save_and_show(fig, PANEL_U)

## Variant: half-width x quarter-height, Hnrnpa2b1 only

A single-gene version of the panel showing only the #1 shortening exemplar,
Hnrnpa2b1 (stacked-area of 3'UTR-site proportions + the per-cell W(t)
trajectory). Sized at `HALF_W_IN x QUARTER_H_IN` (85 x 58.75 mm) so it tiles as
one half of a full-width row, next to `Fig1/fig1_neuron_global_w__quarter`
(same HALF_W x QUARTER_H). Saved as `fig1e_exemplars__hnrnpa2b1_half`.


In [ ]:
# --- Half-width x quarter-height variant: Hnrnpa2b1 only (the #1 shortening gene) ---
# Single-gene version of the panel, sized to tile as one half of a full-width row next to
# fig1_neuron_global_w__quarter (same HALF_W x QUARTER_H). Reuses the helpers,
# `adata_smooth` and `EXEMPLAR_GENES` defined above.
PANEL_H1 = "fig1e_exemplars__hnrnpa2b1_half"
GENE_H1 = "Hnrnpa2b1"
info = EXEMPLAR_GENES[GENE_H1]

fig, (ax_stack, ax_wt) = plt.subplots(
    1,
    2,
    figsize=(S.HALF_W_IN, S.QUARTER_H_IN),  # 85 x 58.75 mm (half W x quarter H)
    gridspec_kw={"width_ratios": [1.3, 1], "wspace": 0.42},
)
CELLTYPE_ORDER = S.MOUSE_LINEAGE_ORDER
df_obs_smooth = adata_smooth.obs[["dpt_pseudotime", "celltype"]].dropna(subset=["dpt_pseudotime"])

props, utr_sizes, site_names, pt = _get_gene_data(adata_smooth, GENE_H1)
assert props is not None, f"{GENE_H1} not found in smoothed AnnData"
order = np.argsort(utr_sizes)
utr_sizes = utr_sizes[order]
site_names = [site_names[i] for i in order]
props = props[:, order]
K = props.shape[1]
colors = _utr_colormap(K)
bin_centres, mean_props, _, _ = _bin_pseudotime(pt, props, n_bins=40)

# stacked area of UTR-site proportions
ax_stack.stackplot(bin_centres, mean_props.T, labels=site_names, colors=colors, alpha=0.85)
ax_stack.set_ylim(-0.14, 1.0)
ax_stack.set_xlim(bin_centres.min(), bin_centres.max())
ax_stack.set_ylabel("Proportion", fontsize=S.FONT_LABEL)
dir_col = S.COL_SHORT if info["direction"] == "Shortening" else S.COL_LONG
ax_stack.set_title(
    f"$\\it{{{GENE_H1}}}$ ({info['direction']})",
    fontsize=S.FONT_LABEL,
    pad=3,
    color=dir_col,
    fontweight="bold",
)
ax_stack.legend(
    loc="upper left",
    fontsize=S.FONT_LEGEND - 0.5,
    framealpha=0.7,
    title=f"K = {K}",
    title_fontsize=S.FONT_LEGEND_TITLE - 0.5,
    handlelength=1,
    handleheight=0.7,
)

# cell-type annotation bar below the stacked area
bar_y0, bar_y1 = -0.13, -0.02
for ct in CELLTYPE_ORDER:
    if ct not in df_obs_smooth["celltype"].values:
        continue
    ct_pt = df_obs_smooth.loc[df_obs_smooth["celltype"] == ct, "dpt_pseudotime"]
    ax_stack.fill_between(
        [float(ct_pt.min()), float(ct_pt.max())],
        bar_y0,
        bar_y1,
        color=S.paper_palette_celltypes[ct],
        alpha=0.9,
        linewidth=0,
    )
ax_stack.axhline(0, color="grey", linewidth=0.5, alpha=0.3)
ax_stack.set_xticks([])
ax_stack.tick_params(axis="x", bottom=False, labelbottom=False)
ax_stack.spines["bottom"].set_visible(False)

# cell-type legend along the bottom (doubles as the pseudotime axis key)
present = [ct for ct in CELLTYPE_ORDER if ct in df_obs_smooth["celltype"].values]
ct_handles = [
    plt.Rectangle((0, 0), 1, 1, fc=S.paper_palette_celltypes[ct], alpha=0.9, linewidth=0)
    for ct in present
]
fig.legend(
    ct_handles,
    present,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncol=len(present),
    fontsize=S.FONT_LEGEND - 0.5,
    frameon=False,
    handlelength=1.0,
    handleheight=0.6,
    columnspacing=0.6,
)

# W(t) trajectory with SEM ribbon
ranks = (rankdata(utr_sizes, method="average") - 1) / (K - 1) if K > 1 else np.array([0.5])
valid = ~np.isnan(pt)
pt_v, props_v = pt[valid], props[valid]
bins_idx = pd.cut(pt_v, bins=40, labels=False)
W_bins, W_means, W_sems = [], [], []
for b in range(40):
    mask = bins_idx == b
    n = mask.sum()
    if n > 0:
        W_cell = props_v[mask] @ ranks
        W_bins.append(pt_v[mask].mean())
        W_means.append(W_cell.mean())
        W_sems.append(W_cell.std(ddof=1) / np.sqrt(n) if n > 1 else 0)
W_bins = np.array(W_bins)
W_means = np.array(W_means)
W_sems = np.array(W_sems)

ax_wt.fill_between(W_bins, W_means - W_sems, W_means + W_sems, alpha=0.25, color=dir_col)
ax_wt.plot(W_bins, W_means, "-o", color=dir_col, markersize=2, linewidth=1.2)
ax_wt.annotate(
    "",
    xy=(W_bins[-1], W_means[-1]),
    xytext=(W_bins[0], W_means[0]),
    arrowprops=dict(arrowstyle="->", color=dir_col, lw=1, alpha=0.4),
)
ax_wt.text(
    0.97,
    0.95,
    f"Δψ = {info['delta_psi_str']}\npadj {info['padj_str']}",
    transform=ax_wt.transAxes,
    ha="right",
    va="top",
    fontsize=S.FONT_ANNOT - 0.5,
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="0.7", boxstyle="round,pad=0.3"),
)
ax_wt.set_xlabel("Pseudotime", fontsize=S.FONT_LABEL)
ax_wt.set_ylabel("W(t)", fontsize=S.FONT_LABEL)
ax_wt.tick_params(labelsize=S.FONT_TICK)

fig.subplots_adjust(left=0.115, right=0.985, top=0.86, bottom=0.26)
S.save_and_show(fig, PANEL_H1)

## Variant: half-width x quarter-height, Hnrnpa2b1 only (uniform axis)

The same single-gene half-panel as `fig1e_exemplars__hnrnpa2b1_half`, but on the
equal-cell (empirical-CDF quantile) pseudotime axis of the `__uniform` panel above,
so cells are uniformly distributed on x. Same size (`HALF_W_IN x QUARTER_H_IN`,
85 x 58.75 mm) and layout, so it drops into the same slot. Saved as
`fig1e_exemplars__hnrnpa2b1_half_uniform`.


In [ ]:
# --- Half-width x quarter-height variant: Hnrnpa2b1 only, uniform (equal-cell) axis ---
# Same single-gene half-panel as fig1e_exemplars__hnrnpa2b1_half, but on the equal-cell
# (empirical-CDF quantile) pseudotime axis from the __uniform panel above. Reuses the
# helpers, `adata_smooth`, `EXEMPLAR_GENES`, and `u_all` / `obs_u` defined above.
GENE_H1U = "Hnrnpa2b1"
PANEL_H1U = "fig1e_exemplars__hnrnpa2b1_half_uniform"
info = EXEMPLAR_GENES[GENE_H1U]

fig, (ax_stack, ax_wt) = plt.subplots(
    1,
    2,
    figsize=(S.HALF_W_IN, S.QUARTER_H_IN),  # 85 x 58.75 mm (half W x quarter H)
    gridspec_kw={"width_ratios": [1.3, 1], "wspace": 0.42},
)
CELLTYPE_ORDER = S.MOUSE_LINEAGE_ORDER

props, utr_sizes, site_names, _ = _get_gene_data(adata_smooth, GENE_H1U)
assert props is not None, f"{GENE_H1U} not found in smoothed AnnData"
order = np.argsort(utr_sizes)
utr_sizes = utr_sizes[order]
site_names = [site_names[i] for i in order]
props = props[:, order]
K = props.shape[1]
colors = _utr_colormap(K)
bin_centres, mean_props, _, _ = _bin_pseudotime(u_all, props, n_bins=40)

# stacked area of UTR-site proportions on the equal-cell quantile axis
ax_stack.stackplot(bin_centres, mean_props.T, labels=site_names, colors=colors, alpha=0.85)
ax_stack.set_ylim(-0.14, 1.0)
ax_stack.set_xlim(0, 1)
ax_stack.set_ylabel("Proportion", fontsize=S.FONT_LABEL)
dir_col = S.COL_SHORT if info["direction"] == "Shortening" else S.COL_LONG
ax_stack.set_title(
    f"$\\it{{{GENE_H1U}}}$ ({info['direction']})",
    fontsize=S.FONT_LABEL,
    pad=3,
    color=dir_col,
    fontweight="bold",
)
ax_stack.legend(
    loc="upper left",
    fontsize=S.FONT_LEGEND - 0.5,
    framealpha=0.7,
    title=f"K = {K}",
    title_fontsize=S.FONT_LEGEND_TITLE - 0.5,
    handlelength=1,
    handleheight=0.7,
)

# cell-type annotation bar (equal-cell u-ranges) below the stacked area
bar_y0, bar_y1 = -0.13, -0.02
for ct in CELLTYPE_ORDER:
    cu = obs_u.loc[obs_u["celltype"] == ct, "u"].dropna()
    if len(cu) == 0:
        continue
    ax_stack.fill_between(
        [float(cu.min()), float(cu.max())],
        bar_y0,
        bar_y1,
        color=S.paper_palette_celltypes[ct],
        alpha=0.9,
        linewidth=0,
    )
ax_stack.axhline(0, color="grey", linewidth=0.5, alpha=0.3)
ax_stack.set_xticks([])
ax_stack.tick_params(axis="x", bottom=False, labelbottom=False)
ax_stack.spines["bottom"].set_visible(False)

# cell-type legend along the bottom (doubles as the pseudotime axis key)
present = [ct for ct in CELLTYPE_ORDER if (obs_u["celltype"] == ct).any()]
ct_handles = [
    plt.Rectangle((0, 0), 1, 1, fc=S.paper_palette_celltypes[ct], alpha=0.9, linewidth=0)
    for ct in present
]
fig.legend(
    ct_handles,
    present,
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncol=len(present),
    fontsize=S.FONT_LEGEND - 0.5,
    frameon=False,
    handlelength=1.0,
    handleheight=0.6,
    columnspacing=0.6,
)

# W(t) trajectory on the equal-cell quantile axis
ranks = (rankdata(utr_sizes, method="average") - 1) / (K - 1) if K > 1 else np.array([0.5])
valid = ~np.isnan(u_all)
u_v, props_v = u_all[valid], props[valid]
bins_idx = pd.cut(u_v, bins=40, labels=False)
W_bins, W_means, W_sems = [], [], []
for b in range(40):
    mask = bins_idx == b
    n = mask.sum()
    if n > 0:
        W_cell = props_v[mask] @ ranks
        W_bins.append(u_v[mask].mean())
        W_means.append(W_cell.mean())
        W_sems.append(W_cell.std(ddof=1) / np.sqrt(n) if n > 1 else 0)
W_bins = np.array(W_bins)
W_means = np.array(W_means)
W_sems = np.array(W_sems)

ax_wt.fill_between(W_bins, W_means - W_sems, W_means + W_sems, alpha=0.25, color=dir_col)
ax_wt.plot(W_bins, W_means, "-o", color=dir_col, markersize=2, linewidth=1.2)
ax_wt.annotate(
    "",
    xy=(W_bins[-1], W_means[-1]),
    xytext=(W_bins[0], W_means[0]),
    arrowprops=dict(arrowstyle="->", color=dir_col, lw=1, alpha=0.4),
)
ax_wt.text(
    0.97,
    0.95,
    f"Δψ = {info['delta_psi_str']}\npadj {info['padj_str']}",
    transform=ax_wt.transAxes,
    ha="right",
    va="top",
    fontsize=S.FONT_ANNOT - 0.5,
    bbox=dict(facecolor="white", alpha=0.8, edgecolor="0.7", boxstyle="round,pad=0.3"),
)
ax_wt.set_xlabel("Pseudotime (quantile)", fontsize=S.FONT_LABEL)
ax_wt.set_ylabel("W(t)", fontsize=S.FONT_LABEL)
ax_wt.set_xlim(0, 1)
ax_wt.set_xticks([0, 0.5, 1.0])
ax_wt.tick_params(labelsize=S.FONT_TICK)

fig.subplots_adjust(left=0.115, right=0.985, top=0.86, bottom=0.26)
S.save_and_show(fig, PANEL_H1U)

### Variants: Hnrnpa2b1 half-panels with cell-type boxes under the W(t) subpanel too

Adds a cell-type-majority box strip below the W(t) trajectory (mirroring the stacked-area
subpanel; the W trajectory is compressed slightly to fit, both subpanels share the same axis
height). Three variants: `__wtbox` (non-uniform, both subpanels linear pseudotime),
`__uniform_wtbox` (both equal-cell), and `__uniform_wtbox_ptlinear` (left equal-cell stack,
right W(t) on linear pseudotime).


In [ ]:
# --- Hnrnpa2b1 half-panels: add cell-type boxes under the W(t) subpanel (3 variants) ---
# left_kind / right_kind in {"equalcell","linear"} pick the x-transform of the stack / W(t).
# The W(t) gets a cell-type box strip below it (extended ylim), aligned with the stack's strip.
_ct_cells = adata_smooth.obs["celltype"].astype(str).values
_pt_raw = adata_smooth.obs["dpt_pseudotime"].values.astype(float)
_CT = S.MOUSE_LINEAGE_ORDER


def _xc_of(kind):
    return (_pt_raw, "Pseudotime") if kind == "linear" else (u_all, "Pseudotime (quantile)")


def _ct_ranges(xc):
    out = []
    for ct in _CT:
        m = (_ct_cells == ct) & ~np.isnan(xc)
        if m.sum():
            out.append((ct, float(xc[m].min()), float(xc[m].max())))
    return out


def _make_h1_panel(save_name, left_kind, right_kind):
    info = EXEMPLAR_GENES["Hnrnpa2b1"]
    dir_col = S.COL_SHORT if info["direction"] == "Shortening" else S.COL_LONG
    props, utr_sizes, site_names, _ = _get_gene_data(adata_smooth, "Hnrnpa2b1")
    order = np.argsort(utr_sizes)
    utr_sizes = utr_sizes[order]
    site_names = [site_names[i] for i in order]
    props = props[:, order]
    K = props.shape[1]
    colors = _utr_colormap(K)
    ranks = (rankdata(utr_sizes, method="average") - 1) / (K - 1) if K > 1 else np.array([0.5])
    fig, (ax_stack, ax_wt) = plt.subplots(
        1,
        2,
        figsize=(S.HALF_W_IN, S.QUARTER_H_IN),
        gridspec_kw={"width_ratios": [1.3, 1], "wspace": 0.42},
    )
    # ---- Left: stacked area + cell-type box strip ----
    xcl, _ = _xc_of(left_kind)
    bc, mp, _, _ = _bin_pseudotime(xcl, props, n_bins=40)
    ax_stack.stackplot(bc, mp.T, labels=site_names, colors=colors, alpha=0.85)
    ax_stack.set_ylim(-0.14, 1.0)
    (ax_stack.set_xlim(0, 1) if left_kind == "equalcell" else ax_stack.set_xlim(bc.min(), bc.max()))
    ax_stack.set_ylabel("Proportion", fontsize=S.FONT_LABEL)
    ax_stack.set_title(
        f"$\\it{{Hnrnpa2b1}}$ ({info['direction']})",
        fontsize=S.FONT_LABEL,
        pad=3,
        color=dir_col,
        fontweight="bold",
    )
    ax_stack.legend(
        loc="upper left",
        fontsize=S.FONT_LEGEND - 0.5,
        framealpha=0.7,
        title=f"K = {K}",
        title_fontsize=S.FONT_LEGEND_TITLE - 0.5,
        handlelength=1,
        handleheight=0.7,
    )
    for ct, xmn, xmx in _ct_ranges(xcl):
        ax_stack.fill_between(
            [xmn, xmx],
            -0.13,
            -0.02,
            color=S.paper_palette_celltypes[ct],
            alpha=0.9,
            linewidth=0,
        )
    ax_stack.axhline(0, color="grey", lw=0.5, alpha=0.3)
    ax_stack.set_xticks([])
    ax_stack.tick_params(axis="x", bottom=False, labelbottom=False)
    ax_stack.spines["bottom"].set_visible(False)
    # ---- Right: W(t) trajectory + cell-type box strip below ----
    xcr, _ = _xc_of(right_kind)
    valid = ~np.isnan(xcr)
    xv, pv = xcr[valid], props[valid]
    bi = pd.cut(xv, bins=40, labels=False)
    Wb, Wm, Ws = [], [], []
    for b in range(40):
        mk = bi == b
        nn = mk.sum()
        if nn > 0:
            Wc = pv[mk] @ ranks
            Wb.append(xv[mk].mean())
            Wm.append(Wc.mean())
            Ws.append(Wc.std(ddof=1) / np.sqrt(nn) if nn > 1 else 0)
    Wb, Wm, Ws = map(np.array, (Wb, Wm, Ws))
    ax_wt.fill_between(Wb, Wm - Ws, Wm + Ws, alpha=0.25, color=dir_col)
    ax_wt.plot(Wb, Wm, "-o", color=dir_col, markersize=2, linewidth=1.2)
    ax_wt.annotate(
        "",
        xy=(Wb[-1], Wm[-1]),
        xytext=(Wb[0], Wm[0]),
        arrowprops=dict(arrowstyle="->", color=dir_col, lw=1, alpha=0.4),
    )
    lo = float((Wm - Ws).min())
    hi = float((Wm + Ws).max())
    span = (hi - lo) or 1.0
    for ct, xmn, xmx in _ct_ranges(xcr):
        ax_wt.fill_between(
            [xmn, xmx],
            lo - 0.13 * span,
            lo - 0.02 * span,
            color=S.paper_palette_celltypes[ct],
            alpha=0.9,
            linewidth=0,
        )
    ax_wt.set_ylim(lo - 0.14 * span, hi)
    ax_wt.axhline(lo, color="grey", lw=0.4, alpha=0.3)
    ax_wt.text(
        0.97,
        0.95,
        f"Δψ = {info['delta_psi_str']}\npadj {info['padj_str']}",
        transform=ax_wt.transAxes,
        ha="right",
        va="top",
        fontsize=S.FONT_ANNOT - 0.5,
        bbox=dict(facecolor="white", alpha=0.8, edgecolor="0.7", boxstyle="round,pad=0.3"),
    )
    ax_wt.set_ylabel("W(t)", fontsize=S.FONT_LABEL)
    (ax_wt.set_xlim(0, 1) if right_kind == "equalcell" else ax_wt.set_xlim(Wb.min(), Wb.max()))
    ax_wt.set_xticks([])
    ax_wt.tick_params(axis="x", bottom=False, labelbottom=False)
    ax_wt.spines["bottom"].set_visible(False)
    # ---- cell-type legend (bottom) ----
    present = [ct for ct, *_ in _ct_ranges(xcl)]
    ct_handles = [
        plt.Rectangle((0, 0), 1, 1, fc=S.paper_palette_celltypes[ct], alpha=0.9, linewidth=0)
        for ct in present
    ]
    fig.legend(
        ct_handles,
        present,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.0),
        ncol=len(present),
        fontsize=S.FONT_LEGEND - 0.5,
        frameon=False,
        handlelength=1.0,
        handleheight=0.6,
        columnspacing=0.6,
    )
    fig.subplots_adjust(left=0.115, right=0.985, top=0.86, bottom=0.26)
    S.save_and_show(fig, save_name)


_make_h1_panel("fig1e_exemplars__hnrnpa2b1_half_wtbox", "linear", "linear")
_make_h1_panel("fig1e_exemplars__hnrnpa2b1_half_uniform_wtbox", "equalcell", "equalcell")
_make_h1_panel("fig1e_exemplars__hnrnpa2b1_half_uniform_wtbox_ptlinear", "equalcell", "linear")